# Projeto Final – Analista de Dados (EBAC)
## Acidentes nas rodovias federais de Pernambuco (PRF)
### Etapa 1 – Seleção dos Dados

**Problema:** onde, quando e por que acontecem os acidentes mais graves nas rodovias federais de Pernambuco, e quem são as vítimas?

**Perguntas que vamos responder ao longo do projeto:**
1. Em quais BRs, trechos (km) e municípios os acidentes se concentram?
2. Em que meses, dias da semana e horários ocorrem mais acidentes e mortes?
3. Quais causas e tipos de acidente são mais letais?
4. Qual o perfil das vítimas (idade, sexo, tipo de veículo, condutor/passageiro/pedestre)?
5. Considerando a população, quais municípios têm a maior taxa de acidentes por 100 mil habitantes?

**Fontes (dados públicos):**

| # | Tabela | Fonte | Granularidade |
|---|---|---|---|
| 1 | `ocorrencias` – Acidentes 2025 (agrupados por ocorrência) | [Dados Abertos da PRF](https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dados-abertos-da-prf) | 1 linha por acidente |
| 2 | `pessoas` – Acidentes 2025 (agrupados por pessoa) | [Dados Abertos da PRF](https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dados-abertos-da-prf) | 1 linha por pessoa envolvida |
| 3 | `populacao` – Estimativas da população dos municípios 2025 | [IBGE – Estimativas de População](https://www.ibge.gov.br/estatisticas/sociais/populacao/9103-estimativas-de-populacao.html) | 1 linha por município |

**Relacionamentos:**
- `ocorrencias.id` ↔ `pessoas.id` (1 acidente → N pessoas)
- `ocorrencias.municipio` ↔ `populacao.municipio` (N acidentes → 1 município)

### Como usar este notebook (Google Colab)
1. Baixe os 3 arquivos nos links acima (os da PRF vêm em `.zip`, não precisa descompactar).
2. No Colab, clique no ícone de pasta 📁 à esquerda e arraste os 3 arquivos para lá.
3. Rode as células em ordem (`Ambiente de execução > Executar tudo`).

In [ ]:
import glob, re, unicodedata
import pandas as pd

pd.set_option('display.max_columns', 50)
UF_ALVO = 'PE'

### 1. Localizar os arquivos enviados

In [ ]:
def achar(padroes):
    for p in padroes:
        achados = sorted(glob.glob(p))
        if achados:
            return achados[0]
    raise FileNotFoundError(f'Nenhum arquivo encontrado para {padroes}. Confira se fez o upload.')

arq_ocorrencias = achar(['datatran*.zip', 'datatran*.csv'])
arq_pessoas     = achar(['acidentes*.zip', 'acidentes*.csv'])
arq_populacao   = achar(['POP*.xls*', 'pop*.xls*', '*estimativa*.xls*'])

print('Ocorrências:', arq_ocorrencias)
print('Pessoas:    ', arq_pessoas)
print('População:  ', arq_populacao)

Ocorrências: datatran2025.zip
Pessoas:     acidentes2025.zip
População:   POP2025_20260828.xls


### 2. Carregar as tabelas da PRF e filtrar Pernambuco
Os CSVs da PRF usam `;` como separador, vírgula como decimal e codificação `latin-1`.

In [ ]:
def ler_prf(caminho):
    return pd.read_csv(caminho, sep=';', encoding='latin-1', decimal=',', low_memory=False)

ocorrencias_br = ler_prf(arq_ocorrencias)
pessoas_br     = ler_prf(arq_pessoas)

ocorrencias = ocorrencias_br[ocorrencias_br['uf'] == UF_ALVO].copy()
pessoas     = pessoas_br[pessoas_br['uf'] == UF_ALVO].copy()

print(f'Ocorrências – Brasil: {len(ocorrencias_br):,} | PE: {len(ocorrencias):,}')
print(f'Pessoas     – Brasil: {len(pessoas_br):,} | PE: {len(pessoas):,}')
ocorrencias.head()

Ocorrências – Brasil: 72,529 | PE: 3,013
Pessoas     – Brasil: 194,629 | PE: 7,646


,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop
9,652648,2025-01-01,quarta-feira,17:00:00,PE,407,7.4,AFRANIO,Demais falhas mecânicas ou elétricas,Incêndio,Sem Vítimas,Pleno dia,Crescente,Céu Claro,Simples,Aclive;Curva,Não,2,0,0,0,2,0,0,1,-8.475031,-41.013711,SPRF-PE,DEL06-PE,UOP02-DEL06-PE
21,652750,2025-01-02,quinta-feira,11:57:00,PE,232,16.0,JABOATAO DOS GUARARAPES,Acessar a via sem observar a presença dos outr...,Colisão traseira,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Dupla,Reta,Não,2,0,1,0,1,0,1,2,-8.094142,-35.037166,SPRF-PE,DEL01-PE,UOP01-DEL01-PE
24,652762,2025-01-02,quinta-feira,13:01:00,PE,101,46.5,IGARASSU,Reação tardia ou ineficiente do condutor,Colisão traseira,Sem Vítimas,Pleno dia,Crescente,Sol,Múltipla,Reta,Sim,3,0,0,0,2,1,0,3,-7.874060,-34.905881,SPRF-PE,DEL01-PE,UOP02-DEL01-PE
41,652867,2025-01-03,sexta-feira,04:12:00,PE,101,29.0,IGARASSU,Manobra de mudança de faixa,Colisão traseira,Sem Vítimas,Amanhecer,Decrescente,Vento,Dupla,Reta,Não,2,0,0,0,2,0,0,2,-7.729397,-34.941859,SPRF-PE,DEL01-PE,UOP02-DEL01-PE
46,652889,2025-01-03,sexta-feira,06:30:00,PE,232,276.0,SERTANIA,Carga excessiva e/ou mal acondicionada,Derramamento de carga,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,3,0,0,1,1,1,1,2,-8.370731,-37.261263,SPRF-PE,DEL04-PE,UOP02-DEL04-PE


In [ ]:
pessoas.head()

,id,pesid,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,id_veiculo,tipo_veiculo,marca,ano_fabricacao_veiculo,tipo_envolvido,estado_fisico,idade,sexo,ilesos,feridos_leves,feridos_graves,mortos,latitude,longitude,regional,delegacia,uop
10,652648,1468101,2025-01-01,quarta-feira,17:00:00,PE,407,7.4,AFRANIO,Demais falhas mecânicas ou elétricas,Incêndio,Sem Vítimas,Pleno dia,Crescente,Céu Claro,Simples,Aclive;Curva,Não,1173253,Caminhão,DAF/CF FAC 310,2024,Passageiro,Ileso,57,Masculino,1,0,0,0,-8.475031,-41.013711,SPRF-PE,DEL06-PE,UOP02-DEL06-PE
11,652648,1468100,2025-01-01,quarta-feira,17:00:00,PE,407,7.4,AFRANIO,Demais falhas mecânicas ou elétricas,Incêndio,Sem Vítimas,Pleno dia,Crescente,Céu Claro,Simples,Aclive;Curva,Não,1173253,Caminhão,DAF/CF FAC 310,2024,Condutor,Ileso,61,Masculino,1,0,0,0,-8.475031,-41.013711,SPRF-PE,DEL06-PE,UOP02-DEL06-PE
26,652750,1468357,2025-01-02,quinta-feira,11:57:00,PE,232,16.0,JABOATAO DOS GUARARAPES,Acessar a via sem observar a presença dos outr...,Colisão traseira,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Dupla,Reta,Não,1173450,Caminhão,SCANIA/P 310 B8X2,2013,Condutor,Ileso,25,Masculino,1,0,0,0,-8.094142,-35.037166,SPRF-PE,DEL01-PE,UOP01-DEL01-PE
30,652762,1468316,2025-01-02,quinta-feira,13:01:00,PE,101,46.5,IGARASSU,Reação tardia ou ineficiente do condutor,Colisão traseira,Sem Vítimas,Pleno dia,Crescente,Sol,Múltipla,Reta,Sim,1173423,Caminhão,VW/10.160 DRC 4X2,2017,Condutor,Ileso,26,Masculino,1,0,0,0,-7.874060,-34.905881,SPRF-PE,DEL01-PE,UOP02-DEL01-PE
49,652867,1468659,2025-01-03,sexta-feira,04:12:00,PE,101,29.0,IGARASSU,Manobra de mudança de faixa,Colisão traseira,Sem Vítimas,Amanhecer,Decrescente,Vento,Dupla,Reta,Não,1173660,Caminhão,VW/17.260 CRM 4X2 4P,2020,Condutor,Ileso,34,Masculino,1,0,0,0,-7.729397,-34.941859,SPRF-PE,DEL01-PE,UOP02-DEL01-PE


### 3. Carregar a população do IBGE
A planilha do IBGE tem linhas de título antes do cabeçalho e notas de rodapé no fim.
O código procura a linha do cabeçalho automaticamente.
> Se der erro ao abrir o `.xls`, abra o arquivo no Excel/Google Sheets, salve como `.xlsx` e envie de novo.

In [ ]:
abas = pd.read_excel(arq_populacao, header=None, sheet_name=None)
for nome_aba, bruto in abas.items():
    achou = bruto.astype(str).apply(lambda r: r.str.contains('NOME DO MUN', case=False).any(), axis=1)
    if achou.any():
        linha_cab = achou[achou].index[0]
        print('Aba usada:', nome_aba)
        break
else:
    raise ValueError('Não achei a aba de municípios na planilha do IBGE.')

Aba usada: Municípios


### 4. Visão geral de cada tabela

In [ ]:
# 1) Monta a tabela de população localizando as colunas pelo nome do cabeçalho
cabecalho = bruto.iloc[linha_cab].astype(str).str.upper().str.strip()

def col(trecho):
    achadas = cabecalho[cabecalho.str.contains(trecho, regex=False)]
    if achadas.empty:
        raise ValueError(f'Coluna com "{trecho}" não encontrada. Cabeçalho: {list(cabecalho)}')
    return achadas.index[0]

mapa = {
    col('NOME DO MUN'): 'municipio',
    col('COD. MUNIC'):  'cod_munic',
    col('POPULA'):      'populacao',
    cabecalho[cabecalho == 'UF'].index[0]: 'uf',
}

populacao_br = bruto.iloc[linha_cab + 1:][list(mapa)].rename(columns=mapa)
populacao_br = populacao_br.dropna(subset=['municipio'])

populacao_br['populacao'] = (populacao_br['populacao'].astype(str)
                             .str.replace(r'\(.*\)', '', regex=True)
                             .str.replace(r'[^0-9]', '', regex=True))
populacao_br = populacao_br[populacao_br['populacao'] != '']
populacao_br['populacao'] = populacao_br['populacao'].astype(int)

populacao = populacao_br[populacao_br['uf'] == UF_ALVO].copy()
print(f'Municípios – Brasil: {len(populacao_br):,} | PE: {len(populacao):,}')

# 2) Visão geral das 3 tabelas
def visao_geral(df, nome):
    print(f'===== {nome}: {df.shape[0]:,} linhas x {df.shape[1]} colunas =====')
    resumo = pd.DataFrame({
        'tipo': df.dtypes.astype(str),
        'nulos': df.isna().sum(),
        '% nulos': (df.isna().mean() * 100).round(1),
        'valores_unicos': df.nunique(),
        'exemplo': df.iloc[0] if len(df) else None,
    })
    display(resumo)

visao_geral(ocorrencias, 'ocorrencias')
visao_geral(pessoas, 'pessoas')
visao_geral(populacao, 'populacao')

Municípios – Brasil: 5,571 | PE: 185
===== ocorrencias: 3,013 linhas x 30 colunas =====


,tipo,nulos,% nulos,valores_unicos,exemplo
id,int64,0,0.0,3013,652648
data_inversa,object,0,0.0,365,2025-01-01
dia_semana,object,0,0.0,7,quarta-feira
horario,object,0,0.0,724,17:00:00
uf,object,0,0.0,1,PE
br,int64,0,0.0,12,407
km,float64,0,0.0,1058,7.4
municipio,object,0,0.0,88,AFRANIO
causa_acidente,object,0,0.0,59,Demais falhas mecânicas ou elétricas
tipo_acidente,object,0,0.0,16,Incêndio


===== pessoas: 7,646 linhas x 35 colunas =====


,tipo,nulos,% nulos,valores_unicos,exemplo
id,int64,0,0.0,3013,652648
pesid,int64,0,0.0,7300,1468101
data_inversa,object,0,0.0,365,2025-01-01
dia_semana,object,0,0.0,7,quarta-feira
horario,object,0,0.0,724,17:00:00
uf,object,0,0.0,1,PE
br,int64,0,0.0,12,407
km,float64,0,0.0,1058,7.4
municipio,object,0,0.0,88,AFRANIO
causa_acidente,object,0,0.0,59,Demais falhas mecânicas ou elétricas


===== populacao: 185 linhas x 4 colunas =====


,tipo,nulos,% nulos,valores_unicos,exemplo
municipio,object,0,0.0,185,Abreu e Lima
cod_munic,object,0,0.0,185,00054
populacao,int64,0,0.0,185,104248
uf,object,0,0.0,1,PE


### 5. Checagem dos requisitos mínimos do projeto

In [ ]:
tabelas = {'ocorrencias': ocorrencias, 'pessoas': pessoas, 'populacao': populacao}
colunas_data = {'ocorrencias': 'data_inversa', 'pessoas': 'data_inversa', 'populacao': None}

quadro = pd.DataFrame([{
    'tabela': n,
    'linhas': len(df),
    'colunas': df.shape[1],
    'tem_data': colunas_data[n] is not None and colunas_data[n] in df.columns,
    '>5 colunas': df.shape[1] > 5,
    '>100 linhas': len(df) > 100,
} for n, df in tabelas.items()])
display(quadro)

requisitos = {
    'Pelo menos 3 tabelas': len(tabelas) >= 3,
    'Pelo menos 1 tabela com data': quadro['tem_data'].sum() >= 1,
    'Pelo menos 2 tabelas com mais de 5 colunas': quadro['>5 colunas'].sum() >= 2,
    'Pelo menos 2 tabelas com mais de 100 linhas': quadro['>100 linhas'].sum() >= 2,
}
for req, ok in requisitos.items():
    print(('✅' if ok else '❌'), req)

,tabela,linhas,colunas,tem_data,>5 colunas,>100 linhas
0,ocorrencias,3013,30,True,True,True
1,pessoas,7646,35,True,True,True
2,populacao,185,4,False,False,True


✅ Pelo menos 3 tabelas
✅ Pelo menos 1 tabela com data
✅ Pelo menos 2 tabelas com mais de 5 colunas
✅ Pelo menos 2 tabelas com mais de 100 linhas


### 6. Teste dos relacionamentos (cruzamento entre tabelas)
- **ocorrências ↔ pessoas** pelo `id` do acidente.
- **ocorrências ↔ população** pelo nome do município. Os nomes são padronizados (maiúsculas, sem acento) para casar.

In [ ]:
# 6.1 ocorrências x pessoas
ids_oc = set(ocorrencias['id'])
ids_pe = set(pessoas['id'])
print(f'Acidentes com pessoas cadastradas: {len(ids_oc & ids_pe):,} de {len(ids_oc):,} '
      f'({len(ids_oc & ids_pe) / len(ids_oc):.1%})')
print(f'Média de pessoas por acidente: {len(pessoas) / pessoas["id"].nunique():.2f}')

Acidentes com pessoas cadastradas: 3,013 de 3,013 (100.0%)
Média de pessoas por acidente: 2.54


In [ ]:
# 6.2 ocorrências x população
def padronizar_nome(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode()
    s = re.sub(r"[^A-Za-z ]", ' ', s)
    return re.sub(r'\s+', ' ', s).strip().upper()

ocorrencias['municipio_chave'] = ocorrencias['municipio'].map(padronizar_nome)
populacao['municipio_chave']   = populacao['municipio'].map(padronizar_nome)

mun_oc  = set(ocorrencias['municipio_chave'])
mun_pop = set(populacao['municipio_chave'])
sem_par = sorted(mun_oc - mun_pop)

print(f'Municípios com acidente: {len(mun_oc)} | encontrados no IBGE: {len(mun_oc & mun_pop)}')
print('Sem correspondência (corrigir na Etapa 3):', sem_par if sem_par else 'nenhum 🎉')

Municípios com acidente: 88 | encontrados no IBGE: 87
Sem correspondência (corrigir na Etapa 3): ['BELEM DE SAO FRANCISCO']


### 7. Salvar as tabelas de Pernambuco para as próximas etapas

In [ ]:
ocorrencias.to_csv('pe_ocorrencias.csv', index=False)
pessoas.to_csv('pe_pessoas.csv', index=False)
populacao.to_csv('pe_populacao.csv', index=False)
print('Arquivos salvos: pe_ocorrencias.csv, pe_pessoas.csv, pe_populacao.csv')

# Para baixar para o computador, descomente:
# from google.colab import files
# for f in ['pe_ocorrencias.csv', 'pe_pessoas.csv', 'pe_populacao.csv']:
#     files.download(f)

Arquivos salvos: pe_ocorrencias.csv, pe_pessoas.csv, pe_populacao.csv


### Conclusão da Etapa 1
- Base escolhida: **acidentes da PRF em Pernambuco (2025)** + **população municipal do IBGE (2025)**.
- As 3 tabelas cumprem os requisitos mínimos (ver quadro da seção 5).
- Os cruzamentos funcionam por `id` (ocorrência ↔ pessoa) e por município (ocorrência ↔ população).
- Próxima etapa: **Análise Exploratória**.